# 🕰️ Lineage splitters: date and provenance-aware holdouts

Welcome! The `lineage` family holds out data by **when** or **where** it came from, rather than by chemical structure. Reach for it whenever the deployment question is *"will this model still work next year / for the next lab / for a partner who isn't in the training data"* — a question a scaffold or similarity split can't answer.

**Contents**
1. [🗓️ `TemporalSplitter`](#1) — real registration/deposition dates
2. [🧬 `SIMPDSplitter`](#2) — a GA-searched pseudo-time split for undated data
3. [🏷️ `SourceSplitter`](#3) — document/lab/vendor provenance
4. [🤝 `PartySplitter`](#4) — federated, leave-one-owner-out evaluation

The examples below share this setup — silencing RDKit's standardizer log spam, and a small helper to print a `SplitResult` compactly:

In [1]:
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

import numpy as np

from chemsplit.datasets import make_dated_series, make_scaffold_families
from chemsplit.splitters.lineage import (
    PartySplitter,
    SIMPDSplitter,
    SourceSplitter,
    TemporalSplitter,
)


def summarize(result, extra=None):
    print(f"train={len(result.train)} valid={len(result.valid)} test={len(result.test)} "
          f"discard={len(result.discard)}  n_records={result.n_records}")
    if extra:
        for k in extra:
            print(f"  metadata[{k!r}] = {result.metadata.get(k)!r}")

<a id="1"></a>
## 1. 🗓️ `TemporalSplitter`

Date-cut split: train on the past, test on the future — the closest available proxy for prospective performance, since it reproduces the real, entangled correlations between chemistry, assay protocol, project goals, and era that a deployed model actually meets.

> 💡 **Advantages**
> - The closest available proxy for prospective performance, since it reproduces the real, entangled correlations between chemistry, assay protocol, project goals, and era that a deployed model actually meets.
> - Needs no featurization and no parameters beyond the cut date, so there's nothing to tune or argue about.
> - `embargo` closes the look-ahead leak from assays reporting months after registration — invisible in a naive date cut.
> - `mode="rolling"`/`"expanding"` produces a performance-over-time curve, which is what a maintenance decision actually needs.
>
> ⚠️ **Pitfalls**
> - Confounds several shifts at once — chemistry, assay protocol, target selection, data volume — so a drop shows the model degrades, not *why*.
> - Dates are frequently wrong: registration, first-test, publication, and deposition dates differ, and datasets often mix them.
> - Public datasets rarely carry usable timestamps. Use `simpd` when dates are absent rather than fabricating them.
> - The test set is one contiguous era, so it's chemically homogeneous with correlated errors; prefer `mode="rolling"`.

| Parameter | Meaning |
|---|---|
| `cut_date` | explicit train/test boundary date; defaults to the size-target quantile of the data |
| `embargo` | records dated within this window after the cut are discarded, not test (closes look-ahead leak) |
| `mode` | `"single"` (one cut), `"rolling"` (sliding windows), or `"expanding"` (growing train set) |
| `n_windows` / `window` | how many folds and how wide each is, for `"rolling"`/`"expanding"` |
| `tie_policy` | where records exactly on the cut date go: `"train"`, `"test"`, or `"discard"` |

In [2]:
fx = make_dated_series(n=300, seed=0)

sp = TemporalSplitter(random_state=0)
result = sp.split_result(fx.smiles, dates=fx.dates)[0]
summarize(result)
print("train max date <= test min date:",
      fx.dates[result.train].max() <= fx.dates[result.test].min())

train=240 valid=0 test=60 discard=0  n_records=300
train max date <= test min date: True


`mode="rolling"` produces several chronological folds instead, useful for a performance-over-time curve:

In [3]:
sp_roll = TemporalSplitter(mode="rolling", n_windows=3, window="200D", random_state=0)
results = sp_roll.split_result(fx.smiles, dates=fx.dates)
for r in results:
    summarize(r, extra=["window_index"])

train=30 valid=0 test=42 discard=228  n_records=300
  metadata['window_index'] = 0
train=42 valid=0 test=39 discard=219  n_records=300
  metadata['window_index'] = 1
train=39 valid=0 test=39 discard=222  n_records=300
  metadata['window_index'] = 2


<a id="2"></a>
## 2. 🧬 `SIMPDSplitter`

Simulated time split: a multi-objective GA rearranges an undated dataset until the train/test pair reproduces the descriptor/property shifts measured in real time splits. Requires the `ga` extra (`deap`).

> 💡 **Advantages**
> - Makes a time-like evaluation possible on the (most) public datasets that lack usable dates.
> - Objectives are explicit and measurable, so "this split resembles a real time split" is checkable via `metadata["achieved"]` vs. `targets`, not just asserted.
> - Multi-objective optimisation surfaces trade-offs instead of collapsing them into one hand-weighted score.
>
> ⚠️ **Pitfalls**
> - **Simulates the statistics of a time split, not time itself.** A model can score well here and still fail prospectively.
> - Default targets are medians from one published study — not universal constants.
> - Expensive: hundreds of generations over hundreds of individuals.
> - Stochastic with conflicting objectives — different seeds give different splits with similar objective values. Report the seed and `metadata["achieved"]`.

| Parameter | Meaning |
|---|---|
| `population_size` / `n_generations` | GA search budget |
| `early_stop_patience` | stop early once the best objective stalls for this many generations |
| `targets` | dict of descriptor/property shifts the GA tries to reproduce (defaults to published medians) |

In [4]:
fx2 = make_scaffold_families(n_scaffolds=10, per_scaffold=20, seed=0)  # n=200
y = np.random.default_rng(0).standard_normal(len(fx2.smiles))

sp_simpd = SIMPDSplitter(population_size=24, n_generations=10, early_stop_patience=4, random_state=0)
result = sp_simpd.split_result(fx2.smiles, y=y)[0]
summarize(result)
print("achieved targets:", sorted(result.metadata["achieved"]))

train=160 valid=0 test=40 discard=0  n_records=200
achieved targets: ['delta_MolLogP', 'delta_MolWt', 'delta_TPSA', 'delta_active_frac', 'frac_test_in_train_cluster', 'g_sim']


The GA's whole job is to reproduce *specified* descriptor/property shifts — passing custom `targets` (a subset of `frac_test_in_train_cluster`, `delta_active_frac`, `delta_MolWt`, `delta_MolLogP`, `delta_TPSA`, `g_sim`) and `descriptors` tailors it to a therapeutic area instead of the published-study defaults, which the docstring is explicit are not universal constants.

In [5]:
sp_simpd_custom = SIMPDSplitter(
    targets={"delta_MolWt": 0.4, "delta_TPSA": 0.2},
    descriptors=("MolWt", "MolLogP", "TPSA"),
    population_size=24, n_generations=10, early_stop_patience=4, random_state=0,
)
result_custom = sp_simpd_custom.split_result(fx2.smiles, y=y)[0]
summarize(result_custom)
print("requested targets:", sp_simpd_custom.targets)
print("achieved:", {k: round(v, 3) for k, v in result_custom.metadata["achieved"].items()})

train=160 valid=0 test=40 discard=0  n_records=200
requested targets: {'frac_test_in_train_cluster': 0.6, 'delta_active_frac': -0.03, 'delta_MolWt': 0.4, 'delta_MolLogP': 0.15, 'delta_TPSA': 0.2, 'g_sim': 0.35}
achieved: {'frac_test_in_train_cluster': 0.15, 'delta_active_frac': -0.094, 'delta_MolWt': -0.177, 'delta_MolLogP': -0.19, 'delta_TPSA': -0.007, 'g_sim': 0.508}


<a id="3"></a>
## 3. 🏷️ `SourceSplitter`

Groups by provenance: document, assay, lab, vendor, plate, or any caller-supplied key. A source (all of its records) is never split across train/test.

> 💡 **Advantages**
> - Catches a leak scaffold and cluster splits both miss: one publication contributing a congeneric series measured under one protocol with one systematic offset.
> - Needs no chemistry, featurization, or seed — it's a metadata join.
> - Composes naturally with `leave_one_cluster_out` (leave-one-source-out) for a per-laboratory error profile.
>
> ⚠️ **Pitfalls**
> - Source metadata is frequently wrong, missing, or inconsistently populated — read `metadata["n_missing_source"]`.
> - Source sizes are extremely skewed, so the realised ratio drifts and `SizeToleranceWarning` should be expected.
> - Grouping by source does **not** guarantee chemical separation — combine with `group_k_fold` on a merged grouping.
> - The chosen hierarchy level changes the experiment (assay- vs. lab-level grouping).

| Parameter | Meaning |
|---|---|
| `source` | per-record provenance key (document id, lab, vendor, ...) |
| `min_source_size` | sources smaller than this are handled per `small_source_policy` |
| `small_source_policy` | `"own_group"`, `"pool"` (merge into one bucket), or `"discard"` |

In [6]:
fx3 = make_scaffold_families(n_scaffolds=10, per_scaffold=10, seed=0)  # n=100
source = [i % 12 for i in range(len(fx3.smiles))]

sp_source = SourceSplitter(source=source, random_state=0)
result = sp_source.split_result(fx3.smiles)[0]
summarize(result, extra=["n_sources"])

train_sources = {source[i] for i in result.train}
test_sources = {source[i] for i in result.test}
print("train/test sources disjoint:", not (train_sources & test_sources))

train=84 valid=0 test=16 discard=0  n_records=100
  metadata['n_sources'] = 12
train/test sources disjoint: True


Real provenance data is rarely evenly sized — a handful of large campaigns plus a long tail of single-record sources. `min_source_size`/`small_source_policy` control what happens to that tail: pool it into one bucket, discard it, or (the default) let each tiny source stand as its own group.

In [7]:
skewed_source = [0] * 70 + [1] * 20 + list(range(2, 12))  # two big sources + 10 singleton sources, n=100
sp_source_pooled = SourceSplitter(
    source=skewed_source, min_source_size=3, small_source_policy="pool", random_state=0,
)
result_pooled = sp_source_pooled.split_result(fx3.smiles)[0]
summarize(result_pooled)
print("n_sources:", result_pooled.metadata["n_sources"], "| source_sizes:", result_pooled.metadata["source_sizes"])

train=80 valid=0 test=20 discard=0  n_records=100
n_sources: 3 | source_sizes: [70, 20, 10]


<a id="4"></a>
## 4. 🤝 `PartySplitter`

Partitions across data owners for federated evaluation, with deliberately non-IID parties (leave-one-party-out).

> 💡 **Advantages**
> - The only way to check whether a federated or consortium model actually helps *each* participant, not just the largest contributor.
> - Synthesis modes let a public dataset stand in for a consortium with an explicit, tunable non-IID severity.
> - `chemical_overlap_matrix` quantifies how different the parties really are.
>
> ⚠️ **Pitfalls**
> - **Never report a pooled average across parties** — it's dominated by the largest party. Report per-party scores; `party_sizes` lets readers weight them.
> - Synthesised parties model heterogeneity, not real heterogeneity.
> - `dirichlet_alpha` has no natural value and must be reported.
> - Says nothing about the privacy properties of the training scheme — this is a data split, not a privacy guarantee.

| Parameter | Meaning |
|---|---|
| `party` | per-record data-owner key |
| `n_parties` | number of distinct parties expected |
| `held_out_party` | which party to hold out as test, or `"each"` for leave-one-party-out folds |

In [8]:
fx4 = make_scaffold_families(n_scaffolds=10, per_scaffold=20, seed=0)  # n=200
party = [i % 4 for i in range(len(fx4.smiles))]

sp_party = PartySplitter(party=party, n_parties=4, held_out_party="each", random_state=0)
print("n_splits:", sp_party.get_n_splits())
results = sp_party.split_result(fx4.smiles)
for r in results:
    summarize(r, extra=["held_out_party"])

n_splits: 4


train=150 valid=0 test=50 discard=0  n_records=200
  metadata['held_out_party'] = 0
train=150 valid=0 test=50 discard=0  n_records=200
  metadata['held_out_party'] = 1
train=150 valid=0 test=50 discard=0  n_records=200
  metadata['held_out_party'] = 2
train=150 valid=0 test=50 discard=0  n_records=200
  metadata['held_out_party'] = 3


No real `party` column? `synthesis="dirichlet"` synthesises non-IID parties directly from the chemistry (Butina clusters split across parties via a Dirichlet draw), letting a public dataset stand in for a consortium. `dirichlet_alpha` sets the severity — low `alpha` means each party gets its own narrow slice of chemical space (harshly non-IID), high `alpha` approaches an even, IID-like split. It has no natural default and must be reported.

In [9]:
sp_party_synth = PartySplitter(
    n_parties=4, synthesis="dirichlet", dirichlet_alpha=0.3, held_out_party="each", random_state=0,
)
results_synth = sp_party_synth.split_result(fx4.smiles)
for r in results_synth:
    summarize(r, extra=["held_out_party"])

train=156 valid=0 test=44 discard=0  n_records=200
  metadata['held_out_party'] = 0
train=151 valid=0 test=49 discard=0  n_records=200
  metadata['held_out_party'] = 1
train=146 valid=0 test=54 discard=0  n_records=200
  metadata['held_out_party'] = 2
train=147 valid=0 test=53 discard=0  n_records=200
  metadata['held_out_party'] = 3


That covers all four `lineage` splitters. Pair `TemporalSplitter`/`SIMPDSplitter` with a structural split (`scaffold`/`similarity` family) rather than using either alone — provenance and structure leak through different, largely independent channels.